# Lab 15: Q-Learning on FrozenLake

**Module 15** | Read `notes/15-rl-mdps-qlearning.pdf` before running this notebook.

Tabular Q-learning with epsilon-greedy exploration on FrozenLake-v1 and render a solved episode.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Tabular Q-learning on FrozenLake

FrozenLake-v1 is a 4×4 grid world: start at top-left, goal at bottom-right, holes terminate the episode. With `is_slippery=False` actions are deterministic, which makes tabular Q-learning reliable.

Q-learning updates action values from experience without a model of the environment. Epsilon-greedy exploration balances trying random moves (discovering the goal) with exploiting the current best-known actions.


### Environment and hyperparameters

There are 16 states and 4 actions (left, down, right, up). We train for 10,000 episodes with a decaying exploration rate is optional; here we keep epsilon fixed for simplicity.


In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=False)
nS = env.observation_space.n
nA = env.action_space.n
Q = np.zeros((nS, nA))

ALPHA = 0.8
GAMMA = 0.99
EPSILON = 0.1
N_EPISODES = 10_000

print(f"States: {nS}, Actions: {nA}")
print(f"Training for {N_EPISODES} episodes (epsilon-greedy, eps={EPSILON})")


### Q-learning loop

Each step applies the Bellman update: move the Q-value toward the observed reward plus discounted best next-state value.


In [ ]:
rewards_per_episode: list[float] = []

for episode in range(N_EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0.0

    while not done:
        if np.random.random() < EPSILON:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

        best_next = np.max(Q[next_state])
        Q[state, action] += ALPHA * (reward + GAMMA * best_next - Q[state, action])
        state = next_state

    rewards_per_episode.append(total_reward)

success_rate = 100 * np.mean(rewards_per_episode[-1000:])
print(f"Success rate (last 1000 episodes): {success_rate:.1f}%")


### Learned policy grid

The greedy policy picks `argmax_a Q(s, a)` per state. Holes and the goal are shown as-is from the map; arrows show the preferred move elsewhere.


In [ ]:
ACTION_SYMBOLS = {0: "←", 1: "↓", 2: "→", 3: "↑"}
MAP_LAYOUT = [
    "SFFF",
    "FHFH",
    "FFFH",
    "HFFG",
]

policy = np.argmax(Q, axis=1)
print("Learned policy (4×4 grid, row-major):")
print("-" * 13)
for row_idx, row in enumerate(MAP_LAYOUT):
    cells = []
    for col_idx, ch in enumerate(row):
        state = row_idx * 4 + col_idx
        if ch in "SHG":
            cells.append(f" {ch} ")
        else:
            cells.append(f" {ACTION_SYMBOLS[policy[state]]} ")
    print("|".join(cells))
print("-" * 13)
print("S=start  G=goal  H=hole  arrows=greedy action")


### Render one solved episode

We replay the greedy policy from the start. Text rendering works in notebooks without opening a separate window.


In [ ]:
render_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
state, _ = render_env.reset()
print(render_env.render())
steps = 0
done = False

while not done and steps < 20:
    action = int(np.argmax(Q[state]))
    state, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    steps += 1
    print(f"Step {steps}: action={ACTION_SYMBOLS[action]} reward={reward}")
    print(render_env.render())

if reward == 1.0:
    print(f"Reached the goal in {steps} steps.")
else:
    print("Episode ended without reaching the goal (unexpected with deterministic map).")

render_env.close()
